#

# Macro Variable convariance + PCA

- Pull macro data from FRED API and form variance-covariance matrix


In [ ]:
import os
import pandas as pd
from fredapi import Fred
from dotenv import load_dotenv
import matplotlib.pyplot as plt

#finds .env and loads up the API Key so os.getenv can see it
load_dotenv()
#create a FRED connection using the API key.
fred = Fred(api_key=os.getenv('FRED_API_KEY'))

#define function to pull data
def get_series(ticker, name, filepath):
    # load locally if already saved — avoids hitting the API every run
    if os.path.exists(filepath):
        data = pd.read_csv(filepath, index_col=0, parse_dates=True).squeeze()
        data.name = name
    else:
        # pull from FRED
        data = fred.get_series(ticker)
        data.index = pd.to_datetime(data.index)
        data.name = name
        
        # check missing before doing anything
        print(f"{name}: {data.isna().sum()} missing values")
        print(data[data.isna()])   # show where gaps are
        
        # forward fill mid-series gaps, drop any remaining at start
        data = data.ffill().dropna()
        data.to_csv(filepath, header=True)
    return data

# pull each series
gdp      = get_series('A191RL1Q225SBEA', 'gdp_growth',    'data/gdp_growth.csv')
indpro   = get_series('INDPRO',          'indpro',         'data/indpro.csv')
payems   = get_series('PAYEMS',          'payems',         'data/payems.csv')
cpi      = get_series('CPIAUCSL',        'cpi',            'data/cpi.csv')
spread   = get_series('BAA10Y',          'credit_spread',  'data/credit_spread.csv')
slope    = get_series('T10Y2Y',          'slope',          'data/slope.csv')

# validate each series — start, end, missing values
for data in [gdp, indpro, payems, cpi, spread, slope]:
    print(f"{data.name}: {data.index[0].date()} → {data.index[-1].date()} | missing: {data.isna().sum()}")

